# 01 — Explore the demo videos and a baseline tracker

Step A2. For the configured take, for each camera:

1. Read the metadata OpenCV and the container header report.
2. Measure the **real** duration and fps from packet timestamps (the browser `.webm` header is unreliable).
3. If needed, convert to constant 30 fps MP4 in `data/raw/converted/<cam_id>_take<N>.mp4` (originals untouched; `paths.py` then prefers the MP4).
4. Check the converted files: fps = 30, duration within 0.2 s of the real duration, cameras within 1 s of each other.
5. Run pre-trained `yolo11n.pt` + ByteTrack (people only) on `cam_1` and report track statistics.

Paths are printed as file names only, because the video root contains a personal Google Drive folder name.

In [1]:
import json, subprocess, sys, time
from collections import Counter
from pathlib import Path

import cv2
import pandas as pd
import torch

REPO = Path.cwd().resolve()
while not (REPO / "data" / "video_config.json").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml" / "tracking"))
from paths import camera_ids, configured_take, converted_path, video_path

TAKE = configured_take()
CAMS = camera_ids()
print("take", TAKE, "| cameras", CAMS)

take 3 | cameras ['cam_1', 'cam_2', 'attendance_cam']


## 1–2. Header metadata vs. real timing of the originals

In [2]:
def ffprobe_json(path, *args):
    out = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", *args, "-of", "json", str(path)],
                         capture_output=True, text=True, check=True).stdout
    return json.loads(out)

def packet_times(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "packet=pts_time",
                          "-of", "csv=p=0", str(path)], capture_output=True, text=True, check=True).stdout
    return [float(x.strip(",")) for x in out.split() if x.strip(",")]

def opencv_meta(path):
    cap = cv2.VideoCapture(str(path))
    meta = dict(cv2_fps=cap.get(cv2.CAP_PROP_FPS), cv2_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
                width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    cap.release()
    return meta

rows = []
for cam in CAMS:
    src = video_path(cam, TAKE, prefer_converted=False)
    hdr = ffprobe_json(src, "-show_entries", "stream=codec_name,r_frame_rate,avg_frame_rate:format=duration")
    pts = packet_times(src)
    real_dur = pts[-1] - pts[0]
    rows.append(dict(cam_id=cam, file=src.name, **opencv_meta(src),
                     codec=hdr["streams"][0]["codec_name"],
                     hdr_r_fps=hdr["streams"][0]["r_frame_rate"], hdr_avg_fps=hdr["streams"][0]["avg_frame_rate"],
                     hdr_duration=hdr["format"].get("duration", "N/A"),
                     packets=len(pts), real_duration_s=round(real_dur, 3),
                     real_fps=round((len(pts) - 1) / real_dur, 2)))
orig = pd.DataFrame(rows).set_index("cam_id")
orig

,file,cv2_fps,cv2_frames,width,height,codec,hdr_r_fps,hdr_avg_fps,hdr_duration,packets,real_duration_s,real_fps
cam_id,,,,,,,,,,,,
cam_1,sCi8Jvd_1794809385941.webm,1000.0,-9223372036854775808,1280,720,vp8,1000/1,0/0,N/A,957,31.365,30.48
cam_2,pDHjQPf_1794809409481.webm,1000.0,-9223372036854775808,1280,720,vp8,1000/1,0/0,N/A,939,31.376,29.90
attendance_cam,2SiEwN8_1794809386098.webm,1000.0,-9223372036854775808,1280,720,vp8,1000/1,0/0,N/A,933,30.991,30.07


In [3]:
# A file is unreliable if OpenCV/header fps is missing or far from the real fps, or the frame count is unusable.
def unreliable(r):
    return (not 1 <= r.cv2_fps <= 120) or abs(r.cv2_fps - r.real_fps) > 2 or r.cv2_frames <= 0 \
        or r.hdr_duration == "N/A"

orig["needs_conversion"] = orig.apply(unreliable, axis=1)
orig[["cv2_fps", "cv2_frames", "hdr_r_fps", "hdr_duration", "real_fps", "real_duration_s", "needs_conversion"]]

,cv2_fps,cv2_frames,hdr_r_fps,hdr_duration,real_fps,real_duration_s,needs_conversion
cam_id,,,,,,,
cam_1,1000.0,-9223372036854775808,1000/1,N/A,30.48,31.365,True
cam_2,1000.0,-9223372036854775808,1000/1,N/A,29.90,31.376,True
attendance_cam,1000.0,-9223372036854775808,1000/1,N/A,30.07,30.991,True


## 3. Convert to constant 30 fps MP4

`-vf fps=30` follows the real timestamps: it drops or duplicates frames to land on a 30 fps grid and keeps the true duration. Audio is dropped. Existing outputs are kept unless `FORCE = True`.

In [4]:
FORCE = False
for cam, r in orig.iterrows():
    out = converted_path(cam, TAKE)
    if not r.needs_conversion:
        print(cam, "reliable, not converted"); continue
    if out.exists() and not FORCE:
        print(cam, "already converted:", out.name); continue
    out.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(video_path(cam, TAKE, prefer_converted=False)),
                    "-vf", "fps=30", "-c:v", "libx264", "-preset", "veryfast", "-crf", "20", "-an", str(out)], check=True)
    print(cam, "converted ->", out.name)

cam_1 already converted: cam_1_take3.mp4
cam_2 already converted: cam_2_take3.mp4
attendance_cam already converted: attendance_cam_take3.mp4


## 4. Check the converted files

In [5]:
rows = []
for cam in CAMS:
    p = converted_path(cam, TAKE)
    j = ffprobe_json(p, "-count_frames", "-show_entries", "stream=r_frame_rate,avg_frame_rate,nb_read_frames:format=duration")
    m = opencv_meta(p)
    dur = float(j["format"]["duration"])
    rows.append(dict(cam_id=cam, file=p.name, ffprobe_fps=j["streams"][0]["avg_frame_rate"],
                     ffprobe_frames=int(j["streams"][0]["nb_read_frames"]), ffprobe_duration_s=round(dur, 3),
                     cv2_fps=m["cv2_fps"], cv2_frames=m["cv2_frames"], cv2_duration_s=round(m["cv2_frames"] / m["cv2_fps"], 3),
                     real_duration_s=orig.loc[cam, "real_duration_s"],
                     diff_s=round(dur - orig.loc[cam, "real_duration_s"], 3)))
conv = pd.DataFrame(rows).set_index("cam_id")
spread = conv.ffprobe_duration_s.max() - conv.ffprobe_duration_s.min()
checks = {
    "fps == 30 (ffprobe and OpenCV)": bool(((conv.ffprobe_fps == "30/1") & (conv.cv2_fps == 30)).all()),
    "duration within 0.2 s of real": bool((conv.diff_s.abs() <= 0.2).all()),
    f"cameras within 1 s of each other (spread {spread:.3f} s)": bool(spread <= 1.0),
}
display(conv)
for k, v in checks.items():
    print("PASS" if v else "FAIL", k)
assert all(checks.values()), "Stop: converted videos failed a check, do not track yet."

,file,ffprobe_fps,ffprobe_frames,ffprobe_duration_s,cv2_fps,cv2_frames,cv2_duration_s,real_duration_s,diff_s
cam_id,,,,,,,,,
cam_1,cam_1_take3.mp4,30/1,942,31.400,30.0,942,31.400,31.365,0.035
cam_2,cam_2_take3.mp4,30/1,942,31.400,30.0,942,31.400,31.376,0.024
attendance_cam,attendance_cam_take3.mp4,30/1,931,31.033,30.0,931,31.033,30.991,0.042


PASS fps == 30 (ffprobe and OpenCV)
PASS duration within 0.2 s of real
PASS cameras within 1 s of each other (spread 0.367 s)


## 5. Baseline tracking on `cam_1`

Pre-trained `yolo11n.pt` (no training), ByteTrack, class 0 (person) only, `conf=0.4`, `persist=True`, `save=True`.
`stream=True` is used so we can count IDs per frame and time the run; it gives the same tracks as a non-streamed call.
The annotated video goes to `runs/track/` at the repo root, which is git-ignored.

In [6]:
CAM = "cam_1"
source = video_path(CAM, TAKE)
assert source == converted_path(CAM, TAKE), "expected the converted MP4"
device = "mps" if torch.backends.mps.is_available() else "cpu"

from ultralytics import YOLO
model = YOLO(str(REPO / "ml" / "tracking" / "yolo11n.pt"))  # weights download here once (git-ignored *.pt)

frames, frames_with_track, ids_per_frame = 0, 0, Counter()
t0 = time.perf_counter()
for result in model.track(source=str(source), classes=[0], tracker="bytetrack.yaml", persist=True, conf=0.4,
                          save=True, stream=True, verbose=False, device=device,
                          project=str(REPO / "runs" / "track"), name=f"{CAM}_take{TAKE}_baseline", exist_ok=True):
    frames += 1
    if result.boxes.id is not None:
        frames_with_track += 1
        ids_per_frame.update(int(i) for i in result.boxes.id.tolist())
elapsed = time.perf_counter() - t0
save_dir = Path(result.save_dir)
print(f"device={device}  frames processed={frames}  wall time={elapsed:.1f} s  effective fps={frames / elapsed:.1f}")

Results saved to /Users/quang/code/Facial_Recognition_TeamProject/runs/track/cam_1_take3_baseline


device=mps  frames processed=942  wall time=47.2 s  effective fps=19.9


In [7]:
per_id = pd.DataFrame(sorted(ids_per_frame.items()), columns=["track_id", "frames_seen"])
per_id["seconds_seen"] = (per_id.frames_seen / 30).round(2)
saved = sorted(p for p in save_dir.iterdir() if p.is_file())
summary = {
    "video": source.name,
    "video duration (s)": conv.loc[CAM, "ffprobe_duration_s"],
    "frames in video": int(conv.loc[CAM, "cv2_frames"]),
    "frames processed": frames,
    "frames with ≥1 tracked person": frames_with_track,
    "unique track IDs": len(per_id),
    "device": device,
    "effective processing fps": round(frames / elapsed, 1),
    "annotated output": [str(p.relative_to(REPO)) for p in saved],
}
display(per_id)
summary

,track_id,frames_seen,seconds_seen
0,1,707,23.57
1,3,106,3.53
2,4,22,0.73
3,5,101,3.37
4,6,321,10.70
5,7,65,2.17
6,8,3,0.10
7,9,389,12.97
8,10,7,0.23
9,11,70,2.33


{'video': 'cam_1_take3.mp4',
 'video duration (s)': np.float64(31.4),
 'frames in video': 942,
 'frames processed': 942,
 'frames with ≥1 tracked person': 936,
 'unique track IDs': 20,
 'device': 'mps',
 'effective processing fps': 19.9,
 'annotated output': ['runs/track/cam_1_take3_baseline/cam_1_take3.mp4']}